# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [1]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

import os
import wandb

/Users/sburtsev/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Объявим класс конфига

In [2]:
class CFG:
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 15
  train_batch_size = 64
  test_batch_size = 256
  lr = 0.001
  seed = 42
  wandb = True

In [ ]:
#конфиг в словарь
def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [ ]:
# вход в вандб , ключ надо вписать 1 раз
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/sburtsev/.netrc.
wandb: Currently logged in as: armntvs-d3v (armntvs-d3v-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Зафиксируем сиды

In [5]:
def seed_everything(seed): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU

In [6]:
# https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
# т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

Загрузим все предобработанные данные 

In [7]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
X_val = pd.read_csv('../data/X_val.csv')

y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')
y_val = pd.read_csv('../data/y_val.csv')


X_train.shape, X_test.shape, X_val.shape, y_train.shape, y_test.shape, y_val.shape

((3707, 570), (795, 570), (794, 570), (3707, 1), (795, 1), (794, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [8]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
X_val = X_val.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)
y_val = y_val.astype(float)

Переведем все в тензоры

In [9]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
X_val_tensor =torch.FloatTensor(X_val.values)

y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)
y_val_tensor =torch.FloatTensor(y_val.values)


X_train_tensor.shape, y_train_tensor.shape

(torch.Size([3707, 570]), torch.Size([3707, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [10]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)
val_ds = TensorDataset(X_val_tensor,y_val_tensor)

train_ds

In [11]:
train_loader = DataLoader(train_ds, batch_size= CFG.train_batch_size, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size= CFG.test_batch_size)
val_loader = DataLoader(val_ds, batch_size= CFG.test_batch_size)


In [12]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [13]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [14]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [15]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [16]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [17]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [18]:
criterion = nn.MSELoss()

In [19]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок

    if WANDB:
        wandb.log({'epoch': epoch,
                   'train_loss': train_loss})
        
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [20]:
def eval(model, device, test_loader, criterion, epoch, WANDB, stage='Test'):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('{} set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
       stage, test_loss, mae, rmse, r2))
    
    if WANDB:
        wandb.log({'epoch': epoch,
                   f'{stage.lower()}_loss': test_loss,
                   f'{stage.lower()}_mae': mae,
                   f'{stage.lower()}_rmse': rmse,
                   f'{stage.lower()}_r2': r2})
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [ ]:
# основная функция для экспериментов
def main(model, model_name):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, name=model_name, reinit=True, config=class2dict(CFG)) #reinit чтобы каждый экспер записывался как новый
        # параметры архитектуры https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})
    seed_everything(CFG.seed)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    
    train_losses = []
    val_losses = []
    
    if CFG.wandb:
        wandb.watch(model, log='all') # логируем все (метрики, лоссы, градиенты)


    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val_loss, val_mae, val_rmse, val_r2 = eval(model, device, val_loader, criterion, epoch, CFG.wandb, stage='Val')
        train_losses.append(train_loss)
        val_losses.append(val_loss)

    print('Training is ended!')
    
    test_loss, test_mae, test_rmse, test_r2 = eval(model, device, test_loader, criterion, CFG.num_epochs, CFG.wandb, stage='Test')

    if CFG.wandb:
        #сохраняем модель https://docs.wandb.ai/guides/artifacts
        os.makedirs('../data/models', exist_ok=True)
        torch.save(model.state_dict(), f'../data/models/{model_name}.pt')
        artifact = wandb.Artifact(model_name, type='model')
        artifact.add_file(f'../data/models/{model_name}.pt')
        wandb.log_artifact(artifact)
        wandb.finish()

    return model, train_losses, val_losses, test_loss, test_mae, test_rmse, test_r2

In [ ]:
#https://docs.wandb.ai/guides/artifacts
#выполнить один раз, далее закомментировать если повторно прогонять
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, name='dataset-tabular', reinit=True)
    art = wandb.Artifact('kolesa-tabular', type='dataset')
    for f in ['X_train.csv', 'X_val.csv', 'X_test.csv', 'y_train.csv', 'y_val.csv', 'y_test.csv']:
        art.add_file(f'../data/{f}')
    wandb.log_artifact(art)
    wandb.finish()

In [23]:
seed_everything(CFG.seed)
simple_model, simple_train_losses, simple_val_losses, simple_test_loss, simple_mae, simple_rmse, simple_r2 = main(Simple(input_size),'Simple')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



Epoch: 1


100%|██████████| 58/58 [00:00<00:00, 219.64it/s]



Train set: Average loss: 177.3284
Val set: Average loss: 13.3810, MAE: 10525731.00, RMSE: 14563566.32, R2: -1.0843

Epoch: 2


100%|██████████| 58/58 [00:00<00:00, 471.96it/s]


...

100%|██████████| 58/58 [00:00<00:00, 541.75it/s]



Train set: Average loss: 0.0790
Val set: Average loss: 0.3734, MAE: 2880294.75, RMSE: 6165751.19, R2: 0.6264

Epoch: 15


100%|██████████| 58/58 [00:00<00:00, 447.27it/s]



Train set: Average loss: 0.0711
Val set: Average loss: 0.3623, MAE: 2820936.25, RMSE: 5967266.85, R2: 0.6501
Training is ended!
Test set: Average loss: 0.2267, MAE: 2598322.50, RMSE: 4808711.20, R2: 0.7929


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇███
test_loss,▁
test_mae,▁
test_r2,▁
test_rmse,▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,▅█▅▄▃▃▂▂▂▁▁▁▁▁▁
val_r2,▇▁▆▆▇▇▇████████
val_rmse,▃█▄▄▄▃▂▂▂▂▁▁▁▁▁
epoch,15


In [24]:
seed_everything(CFG.seed)
deep_model, deep_train_losses, deep_val_losses, deep_test_loss, deep_mae, deep_rmse, deep_r2 = main(Deep(input_size), 'Deep')


Epoch: 1


100%|██████████| 58/58 [00:00<00:00, 259.41it/s]



Train set: Average loss: 80.3491
Val set: Average loss: 4.2232, MAE: 8547291.00, RMSE: 12441121.11, R2: -0.5211

Epoch: 2


100%|██████████| 58/58 [00:00<00:00, 318.16it/s]



Train set: Average loss: 1.2981
Val set: Average loss: 0.9606, MAE: 8697233.00, RMSE: 22018515.75, R2: -3.7643

Epoch: 3


...

100%|██████████| 58/58 [00:00<00:00, 413.35it/s]



Train set: Average loss: 0.0432
Val set: Average loss: 0.2910, MAE: 2645462.25, RMSE: 6081134.51, R2: 0.6366

Epoch: 15


100%|██████████| 58/58 [00:00<00:00, 319.50it/s]



Train set: Average loss: 0.0395
Val set: Average loss: 0.3030, MAE: 2599450.50, RMSE: 5113933.51, R2: 0.7430
Training is ended!
Test set: Average loss: 0.2040, MAE: 2344159.25, RMSE: 4335691.94, R2: 0.8317


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇███
test_loss,▁
test_mae,▁
test_r2,▁
test_rmse,▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,██▅▄▃▃▂▂▂▂▁▁▁▁▁
val_r2,▆▁▄▄▆▇▇▇▇██████
val_rmse,▄█▆▆▄▃▂▃▂▂▁▂▁▁▁
epoch,15


In [25]:
seed_everything(CFG.seed)
regularized_model, regularized_train_losses, regularized_val_losses, regularized_test_loss, regularized_mae, regularized_rmse, regularized_r2 = main(
    Regularized(input_size),'Regularized')


Epoch: 1


100%|██████████| 58/58 [00:00<00:00, 243.62it/s]



Train set: Average loss: 129.9301
Val set: Average loss: 24.8162, MAE: 11141749.00, RMSE: 15005059.11, R2: -1.2126

Epoch: 2


100%|██████████| 58/58 [00:00<00:00, 314.74it/s]



Train set: Average loss: 2.0497
Val set: Average loss: 1.1725, MAE: 6403346.50, RMSE: 9849442.04, R2: 0.0467

Epoch: 3


...

100%|██████████| 58/58 [00:00<00:00, 377.15it/s]



Train set: Average loss: 0.4796
Val set: Average loss: 0.5538, MAE: 5904470.50, RMSE: 8953553.85, R2: 0.2122

Epoch: 15


100%|██████████| 58/58 [00:00<00:00, 342.42it/s]



Train set: Average loss: 0.5039
Val set: Average loss: 0.2074, MAE: 4173671.75, RMSE: 7192190.82, R2: 0.4917
Training is ended!
Test set: Average loss: 0.2532, MAE: 4222111.50, RMSE: 7269826.54, R2: 0.5267


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇███
test_loss,▁
test_mae,▁
test_r2,▁
test_rmse,▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,█▃▂▂▁▄▂▂▁▁▁▂▁▃▁
val_r2,▁▆▇██▃▇▄▇██▇█▇█
val_rmse,█▃▂▁▁▆▃▆▃▁▁▃▁▃▁
epoch,15


In [26]:
seed_everything(CFG.seed)
res_model, res_train_losses, res_val_losses, res_test_loss, res_mae, res_rmse, res_r2 = main(ResMLP(input_size), 'ResMLP')


Epoch: 1


100%|██████████| 58/58 [00:00<00:00, 274.77it/s]



Train set: Average loss: 47.3857
Val set: Average loss: 1.6215, MAE: 8546270.00, RMSE: 23072096.47, R2: -4.2312

Epoch: 2


100%|██████████| 58/58 [00:00<00:00, 312.64it/s]



Train set: Average loss: 0.8326
Val set: Average loss: 0.6123, MAE: 8553997.00, RMSE: 33947867.43, R2: -10.3253

Epoch: 3


...

100%|██████████| 58/58 [00:00<00:00, 310.39it/s]



Train set: Average loss: 0.0325
Val set: Average loss: 0.1962, MAE: 2484277.75, RMSE: 4510751.73, R2: 0.8000

Epoch: 15


100%|██████████| 58/58 [00:00<00:00, 346.47it/s]



Train set: Average loss: 0.0302
Val set: Average loss: 0.1871, MAE: 2459178.00, RMSE: 4477382.13, R2: 0.8030
Training is ended!
Test set: Average loss: 0.1456, MAE: 2431247.25, RMSE: 4377199.84, R2: 0.8284


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇███
test_loss,▁
test_mae,▁
test_r2,▁
test_rmse,▁
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁
val_mae,██▄▃▂▂▂▁▁▁▁▁▁▁▁
val_r2,▅▁▆▇███████████
val_rmse,▅█▄▄▂▁▁▁▁▁▁▁▁▁▁
epoch,15


In [27]:
res = pd.DataFrame([{'model': 'Simple', 'test_loss': simple_test_loss, 'MAE': simple_mae, 'RMSE': simple_rmse, 'R2': simple_r2},
                        {'model': 'Deep', 'test_loss': deep_test_loss, 'MAE': deep_mae, 'RMSE': deep_rmse, 'R2': deep_r2},
                        {'model': 'Regularized', 'test_loss': regularized_test_loss, 'MAE': regularized_mae, 'RMSE': regularized_rmse, 'R2': regularized_r2},
                        {'model': 'ResMLP', 'test_loss': res_test_loss, 'MAE': res_mae, 'RMSE': res_rmse, 'R2': res_r2}])

res

,model,test_loss,MAE,RMSE,R2
0,Simple,0.226656,2598322.50,4.808711e+06,0.792918
1,Deep,0.204008,2344159.25,4.335692e+06,0.831655
2,Regularized,0.253193,4222111.50,7.269827e+06,0.526704
3,ResMLP,0.145645,2431247.25,4.377200e+06,0.828416


In [28]:
res.sort_values('MAE')

,model,test_loss,MAE,RMSE,R2
1,Deep,0.204008,2344159.25,4.335692e+06,0.831655
3,ResMLP,0.145645,2431247.25,4.377200e+06,0.828416
0,Simple,0.226656,2598322.50,4.808711e+06,0.792918
2,Regularized,0.253193,4222111.50,7.269827e+06,0.526704


### Сравнение моделей и выводы

По ходу работы мы обучили 4 модели, от простой базовой полносвязной сети до полносвязной с residual блоками. Основная метрика - считаем MAE, средняя ошибка прогноза цены в тенге. 

По результатам: лучшей стала модель Deep с показателями MAE - примерно 2,34 млн тенге, R2 - примерно 0,83, то есть модель лучше остальных отличает цены и в среднем меньше ошибается. Базовая модель(Simple) и модель с ResMLP также дают хороший результат. Худший результат показала модель Regularized, возможно dropout для нашего датасета мешала модели запоминать какие-то важные зависимости в данных или еще что-то, но любые манипуляции с весами не помогали. 